# Capon Beamforming (MVDR)
## Tutorial 5: Minimum Variance Distortionless Response

Capon beamforming (1969), also known as **MVDR** (Minimum Variance Distortionless Response), adaptively computes weights that **minimise output power** while maintaining unit gain in the look direction.

Topics:
1. **MVDR formulation** – constrained optimisation
2. **Capon spectrum** and its derivation
3. **Diagonal loading** for numerical stability
4. **Adaptive nulling** of interference
5. **Comparison with CBF**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.classical import ConventionalBeamforming, CaponBeamforming

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array  = UniformLinearArray(M=M, d=0.5)
signal_model = SignalModel(array)
cbf    = ConventionalBeamforming(array)
capon  = CaponBeamforming(array)

print("Setup complete.")

## 1. MVDR Formulation

We want weights $\mathbf{w}$ that:

$$\min_{\mathbf{w}} \; \mathbf{w}^H\hat{\mathbf{R}}\mathbf{w} \quad \text{subject to} \quad \mathbf{w}^H\mathbf{a}(\theta) = 1$$

The unique solution (via Lagrange multipliers) is:

$$\mathbf{w}_{\text{MVDR}}(\theta) = \frac{\hat{\mathbf{R}}^{-1}\mathbf{a}(\theta)}{\mathbf{a}^H(\theta)\hat{\mathbf{R}}^{-1}\mathbf{a}(\theta)}$$

The output power (Capon spectrum) is:

$$P_{\text{Capon}}(\theta) = \frac{1}{\mathbf{a}^H(\theta)\hat{\mathbf{R}}^{-1}\mathbf{a}(\theta)}$$

The weights are **data-dependent** — they adapt to suppress interference.

In [ ]:
doas_true = np.deg2rad([-20.0, 15.0])
snr_db = 10
N = 200
angle_grid = np.linspace(-np.pi/2, np.pi/2, 1801)

X, _, _ = signal_model.generate_signals(
    doas=doas_true, N_snapshots=N, snr_db=snr_db, seed=42)

# Both beamformers
bp_cbf   = cbf.beam_pattern(X, angle_grid)
bp_capon = capon.beam_pattern(X, angle_grid)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp_cbf   + 1e-12),
        'b-',  lw=2, label='CBF')
ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp_capon + 1e-12),
        'r-',  lw=2, label='Capon (MVDR)')
for th in doas_true:
    ax.axvline(np.rad2deg(th), color='k', ls='--', alpha=0.6)
ax.set_xlabel('θ (°)'); ax.set_ylabel('Power (dB)')
ax.set_title(f'CBF vs Capon  (M={M}, SNR={snr_db} dB, N={N})')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-40, 10)
plt.tight_layout(); plt.show()

doas_capon = capon.estimate(X, K=len(doas_true))
print(f"True   : {np.round(np.rad2deg(doas_true), 2)} °")
print(f"Capon  : {np.round(np.rad2deg(doas_capon), 2)} °")

## 2. Why Is Capon Better?

Because $\hat{\mathbf{R}}^{-1}$ **pre-whitens** the data.  The matrix inverse places implicit nulls at directions that contribute energy other than from the look direction.  The result is a **much narrower** effective beam.

Capon's spectrum is an exact inverse of the minimum output power — each point is a separate adaptive beamformer.  This gives it approximately **twice the resolution** of CBF for the same aperture.

In [ ]:
separations_deg = [8, 5, 3]
fig, axes = plt.subplots(1, len(separations_deg), figsize=(18, 6), sharey=True)

for ax, sep in zip(axes, separations_deg):
    th1 = -np.deg2rad(sep/2)
    th2 =  np.deg2rad(sep/2)
    X2, _, _ = signal_model.generate_signals(
        doas=[th1, th2], N_snapshots=300, snr_db=15, seed=5)
    bp_c = cbf.beam_pattern(X2, angle_grid)
    bp_m = capon.beam_pattern(X2, angle_grid)
    ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp_c + 1e-12), 'b-',
            lw=2, label='CBF')
    ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp_m + 1e-12), 'r-',
            lw=2, label='Capon')
    ax.axvline(np.rad2deg(th1), color='k', ls='--')
    ax.axvline(np.rad2deg(th2), color='k', ls='--')
    ax.set_title(f'Separation = {sep}°')
    ax.set_xlabel('θ (°)'); ax.set_ylim(-40, 10)
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

axes[0].set_ylabel('Power (dB)')
plt.suptitle('Resolution Comparison: CBF vs Capon', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Diagonal Loading for Robustness

When $N$ is small (especially $N \leq M$), $\hat{\mathbf{R}}$ may be singular or poorly conditioned, and its inverse blows up.  **Diagonal loading** regularises:

$$\hat{\mathbf{R}}_{\delta} = \hat{\mathbf{R}} + \delta\mathbf{I}$$

The parameter $\delta$ is called the **loading level**.  It shrinks the effective inverse back towards uniform weights, providing a smooth transition between Capon and CBF.

In [ ]:
N_small = 20    # fewer snapshots than elements!
X_small, _, _ = signal_model.generate_signals(
    doas=doas_true, N_snapshots=N_small, snr_db=10, seed=99)
R_small = X_small @ X_small.conj().T / N_small
print(f"Condition number of R̂ (N={N_small}): {np.linalg.cond(R_small):.2e}")

loading_factors = [0.0, 0.01, 0.1, 1.0]
fig, ax = plt.subplots(figsize=(13, 6))
colors = ['r','orange','green','blue']

for col, delta in zip(colors, loading_factors):
    cap_loaded = CaponBeamforming(array, diagonal_loading=delta)
    try:
        bp = cap_loaded.beam_pattern(X_small, angle_grid)
        ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp + 1e-12),
                lw=2, color=col, label=f'δ={delta}')
    except Exception as e:
        print(f"δ={delta}: {e}")

for th in doas_true:
    ax.axvline(np.rad2deg(th), color='k', ls='--', alpha=0.5)

ax.set_xlabel('θ (°)'); ax.set_ylabel('Power (dB)')
ax.set_title(f'Diagonal Loading Effect  (N={N_small} < M={M})')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-40, 20)
plt.tight_layout(); plt.show()

## 4. Adaptive Nulling

One of Capon's key strengths: if there is a **strong interferer** it is automatically nulled.

Let us demonstrate with one desired source and one strong interferer.

In [ ]:
theta_desired    = np.deg2rad(0)
theta_interferer = np.deg2rad(30)
snr_signal_db    = 10
snr_interf_db    = 25    # much stronger

X_int, _, _ = signal_model.generate_signals(
    doas=[theta_desired, theta_interferer],
    N_snapshots=500,
    snr_db=[snr_signal_db, snr_interf_db],
    seed=13)

bp_cbf_int   = cbf.beam_pattern(X_int, angle_grid)
bp_capon_int = capon.beam_pattern(X_int, angle_grid)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp_cbf_int   + 1e-12),
        'b-', lw=2, label='CBF')
ax.plot(np.rad2deg(angle_grid), 10*np.log10(bp_capon_int + 1e-12),
        'r-', lw=2, label='Capon')
ax.axvline(np.rad2deg(theta_desired),    color='green', ls='--', lw=2, label='Desired 0°')
ax.axvline(np.rad2deg(theta_interferer), color='purple', ls='--', lw=2,
           label=f'Interferer 30° (+{snr_interf_db-snr_signal_db} dB)')
ax.set_xlabel('θ (°)'); ax.set_ylabel('Power (dB)')
ax.set_title('Adaptive Nulling: Capon vs CBF')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-40, 30)
plt.tight_layout(); plt.show()

## Summary

| Feature | CBF | Capon (MVDR) |
|---|---|---|
| Resolution | Rayleigh limit | ~2× better |
| Sidelobe control | Fixed (window) | Adaptive |
| Robustness to small $N$ | Good | Poor (needs $N \gg M$) |
| Computational cost | Low | Moderate ($O(M^3)$ inversion) |
| Handles correlated sources | Yes | Degrades |

## Exercises
1. Derive the MVDR weights using Lagrange multipliers.
2. Show that as $\delta \to \infty$ in diagonal loading, the Capon weights converge to CBF weights.
3. For $M=16, N=50, \text{SNR}=10$ dB: sweep $\delta$ from $10^{-3}$ to $10^2$ and plot the estimated DOA error vs $\delta$ for two sources at $-20°$ and $15°$.